# Cloud HAND-FIM: Python SDK submission, monitoring, and a COG map

This notebook imports the **generated OpenAPI Python SDK** directly to call a Cloud Run API. No CLI subprocesses are used. HydroShare authentication and an S3 output bucket are required. Jobs incur cloud costs. Run cells in order; never rerun submission to recover from a monitoring failure. Use `RESUME_JOB_ID` instead.

Tokens are entered without echoing and used by AuthenticatedClient. Do not print token variables, client objects, authentication responses, or presigned URLs. See the [deployment instructions](../readme.md). Select a Python 3.10+ kernel.

In [1]:
%pip install httpx rasterio ipyleaflet pillow

from pathlib import Path
import importlib
import site
import sysconfig

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/fimbox_sdk').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the FIMBox checkout, or set ROOT to its absolute path.')
%pip install -e "$ROOT"

# Load editable-install .pth files created after this kernel started.
for package_dir in {sysconfig.get_path('purelib'), sysconfig.get_path('platlib')}:
    site.addsitedir(package_dir)
importlib.invalidate_caches()

from fimbox_sdk import Client, AuthenticatedClient
from fimbox_sdk.api.authentication import authentication_config_auth_config_get, current_session_auth_session_get
from fimbox_sdk.api.default import submit_hand_fim_jobs_hand_fim_post, get_job_status_jobs_job_id_get, get_job_output_jobs_job_id_outputs_index_tif_get
from fimbox_sdk.models import AuthenticationConfig, JobSubmitted, JobStatus, PresignedOutput, Reach, Scenario, StoredHandFimResult


Note: you may need to restart the kernel to use updated packages.
Obtaining file:///Users/scootna/projects/FIMBox
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for fimbox (pyproject.toml) ... done
  Created wheel for fimbox: filename=fimbox-0.1.0-0.editable-py3-none-any.whl size=17967 sha256=8fd83fe467d01c97e1e086f3ca964147fa34b91e5182b475791089461ef6900f
  Stored in directory: /private/var/folders/vs/smnlb2vx3r9f2dlv0_tyndq00000gn/T/pip-ephem-wheel-cache-fzz6zq3t/wheels/2f/95/17/6ea498de9ae922df4c14a06b840d666d32ba03bd9ff645eec5
Successfully built fimbox
  Attempting uninstall: fimbox
    Found existing installation: fimbox 0.1.0
    Uninstalling fimbox-0.1.0:
      Successfully uninstalled fimbox-0.1.0
Note: you may need to restart the kernel to use updated packages.


## Configure the destination
Get the HTTPS endpoint with `terraform -chdir=terraform output -raw api_url` in a terminal. Use your HydroShare bucket, not the Google Artifact Registry repository. Leave `RESUME_JOB_ID` empty only for a new job. `MAX_PROCS=1` limits scenario concurrency; Cloud Run scratch files still consume worker memory.

In [14]:
import os
from urllib.parse import urlsplit

API_URL = os.environ.get('FIMBOX_API_URL', 'https://fimbox-dev-api-307400264918.us-central1.run.app').rstrip('/')
OUTPUT_BUCKET = 'resource'
OUTPUT_PREFIX = 'ad1ae21dec7648efa78d5015c73728d4/data/contents/'
MAX_PROCS = 1
RESUME_JOB_ID = ''
USE_MACOS_KEYCHAIN = False  # Set True to bypass token entry in the notebook.
POLL_SECONDS = 10
WAIT_SECONDS = 1800

origin = urlsplit(API_URL)
if origin.scheme != 'https' or not origin.hostname or origin.username or origin.password or origin.query or origin.fragment or 'YOUR-' in API_URL:
    raise ValueError('Set API_URL to the actual HTTPS API endpoint, without credentials or query parameters.')
if OUTPUT_BUCKET == 'YOUR_HYDROSHARE_BUCKET' or not OUTPUT_BUCKET:
    raise ValueError('Set OUTPUT_BUCKET to your HydroShare S3 bucket.')


## Authenticate
Run the next cell to display the login link and a masked token field. Sign in with HydroShare and paste the callback JSON's **FIMBox `access_token`** into that field (not the provider's access token). Then run the separate Connect cell below. Do not use Run All: pause to enter the token first. An existing FIMBOX_SESSION_TOKEN environment value takes precedence. The SDK does not save tokens.

In [5]:
import ipywidgets as widgets
import html
import httpx
from IPython.display import HTML, display

def call_api(operation, *, client, expected_status=200, expected_type=None, **kwargs):
    try:
        response = operation.sync_detailed(client=client, **kwargs)
    except httpx.RequestError:
        raise RuntimeError('API connection failed. If submitting, inspect your jobs before attempting another submission.') from None
    except (ValueError, KeyError, TypeError, AttributeError):
        raise RuntimeError('API response does not match the generated SDK contract.') from None
    if response.status_code != expected_status:
        raise RuntimeError(f'API request failed (HTTP {response.status_code}). Check endpoint, session expiry, bucket access, and server logs.')
    if expected_type is not None and not isinstance(response.parsed, expected_type):
        raise RuntimeError('API returned an unexpected response model.')
    return response.parsed

with Client(base_url=API_URL, timeout=httpx.Timeout(30), follow_redirects=False) as public_client:
    auth_config = call_api(authentication_config_auth_config_get, client=public_client, expected_type=AuthenticationConfig)
if auth_config.authentication_enabled is not True:
    raise RuntimeError('Expected a reachable, authentication-enabled cloud API.')
display(HTML(f'<a href="{html.escape(API_URL + "/auth/login", quote=True)}" target="_blank" rel="noopener noreferrer">Get a FIMBox session token</a>'))
old_token_input = globals().get('token_input')
if isinstance(old_token_input, widgets.Password):
    old_token_input.value = ''
    old_token_input.close()
del old_token_input
token_input = widgets.Password(description='Session token:', placeholder='Paste FIMBox access_token here',
                               layout=widgets.Layout(width='550px'), style={'description_width': 'initial'})
display(token_input)
print('Paste your token into the field above, then run the Connect cell below.')


Password(description='Session token:', layout=Layout(width='550px'), placeholder='Paste FIMBox access_token he…

Paste your token into the field above, then run the Connect cell below.


### Connect using the entered or saved token

If paste does not work in the widget on macOS, save the token from a terminal at the repository root:
```bash
.venv-1/bin/python -m keyring set fimbox https://YOUR-CLOUD-RUN-ENDPOINT
```
Replace the URL with the exact API_URL (no trailing slash). Paste the token at the terminal's hidden password prompt, not on the command line. Set `USE_MACOS_KEYCHAIN = True` in the configuration cell and rerun that cell, then this Connect cell. This reads native macOS Keychain only, with no plaintext fallback.
Run this cell after filling the field above. It clears the field and validates the session. If the field does not render, enable ipywidgets support in your notebook frontend. Never put the token in notebook source code or print widget state; masked widgets are not a credential store. Clear all outputs/widget state before saving or sharing an executed notebook.

In [7]:
session_token = os.environ.get('FIMBOX_SESSION_TOKEN') or token_input.value
if not session_token and USE_MACOS_KEYCHAIN:
    import sys
    from keyring.errors import KeyringError
    if sys.platform != 'darwin':
        raise RuntimeError('USE_MACOS_KEYCHAIN is available only on macOS.')
    from keyring.backends.macOS import Keyring
    try:
        session_token = Keyring().get_password('fimbox', API_URL)
    except (KeyringError, OSError):
        raise RuntimeError('Cannot read macOS Keychain; check access permissions.') from None
    if not session_token:
        raise ValueError('No saved FIMBox token found for API_URL. Use the keyring set command above first.')
if not session_token.strip():
    raise ValueError('Paste the FIMBox token into the masked field in the previous cell, then rerun this cell.')
token_input.value = ''

previous_client = globals().get('api_client')
if isinstance(previous_client, AuthenticatedClient):
    previous_client.get_httpx_client().close()
del previous_client
api_client = AuthenticatedClient(base_url=API_URL, token=session_token.strip(),
                                 timeout=httpx.Timeout(30), follow_redirects=False)
_ = call_api(current_session_auth_session_get, client=api_client)
print('Authenticated to the cloud API.')







Authenticated to the cloud API.


In [15]:
import tempfile
from uuid import UUID

scenarios = [Scenario(scenario_id='single-reach-scenario', reaches=[
    Reach(huc='03020201', reach_id='8780625', streamflow=7)
])]

submitted = call_api(submit_hand_fim_jobs_hand_fim_post, client=api_client,
                        expected_status=202, expected_type=JobSubmitted, body=scenarios,
                        output_bucket=OUTPUT_BUCKET, output_prefix=OUTPUT_PREFIX, max_procs=MAX_PROCS)
job_id = submitted.job_id
print(f'Job ID: {job_id}')
print('Record this ID for RESUME_JOB_ID if you restart the kernel.')


Job ID: 8c7f4132-b3fe-4477-a3ed-f0a3776a8092
Record this ID for RESUME_JOB_ID if you restart the kernel.


In [16]:
import time

deadline = time.monotonic() + WAIT_SECONDS
while True:
    status_response = call_api(get_job_status_jobs_job_id_get, client=api_client,
                               expected_type=JobStatus, job_id=UUID(job_id))
    state = status_response.status
    print(f'{time.strftime("%H:%M:%S")}  {job_id}  {state}')
    if state == 'success':
        final_result = status_response.result
        if not isinstance(final_result, StoredHandFimResult):
            raise RuntimeError('Successful job has no typed output result.')
        break
    if state in {'failure', 'revoked'}:
        raise RuntimeError(f'Job {job_id} ended with {state}. Inspect fimbox status and worker logs; do not automatically resubmit.')
    remaining = deadline - time.monotonic()
    if remaining <= 0:
        raise TimeoutError(f'Stopped waiting for {job_id}. The cloud job continues; rerun this monitoring cell, not submission.')
    time.sleep(min(POLL_SECONDS, remaining))

for artifact in final_result.files:
    print(artifact.path, artifact.size_bytes, 'bytes')


11:47:53  8c7f4132-b3fe-4477-a3ed-f0a3776a8092  started
11:48:03  8c7f4132-b3fe-4477-a3ed-f0a3776a8092  started
11:48:13  8c7f4132-b3fe-4477-a3ed-f0a3776a8092  started
11:48:23  8c7f4132-b3fe-4477-a3ed-f0a3776a8092  started
11:48:33  8c7f4132-b3fe-4477-a3ed-f0a3776a8092  started
11:48:43  8c7f4132-b3fe-4477-a3ed-f0a3776a8092  started
11:48:54  8c7f4132-b3fe-4477-a3ed-f0a3776a8092  started
11:49:04  8c7f4132-b3fe-4477-a3ed-f0a3776a8092  started
11:49:14  8c7f4132-b3fe-4477-a3ed-f0a3776a8092  started
11:49:24  8c7f4132-b3fe-4477-a3ed-f0a3776a8092  success
8c7f4132-b3fe-4477-a3ed-f0a3776a8092-0.cog 4156 bytes
8c7f4132-b3fe-4477-a3ed-f0a3776a8092-0.tif 5539 bytes


## Download the completed scenario's COG
Request a fresh presigned URL through the SDK. Download directly from S3 **without sending the FIMBox session token**. URLs expire: rerun this cell to get a new one. The notebook downloads the full COG locally; large outputs need sufficient local disk space. No signed URL is displayed.

In [17]:
if 'cog_directory' not in globals():
    cog_directory = tempfile.TemporaryDirectory(prefix='fimbox-cog-')
cog_path = Path(cog_directory.name) / f'{job_id}.cog.tif'
partial_path = cog_path.with_suffix('.part')
signed = call_api(get_job_output_jobs_job_id_outputs_index_tif_get, client=api_client,
                  expected_type=PresignedOutput, job_id=UUID(job_id), index=0)
signed_url = signed.url
if urlsplit(signed_url).scheme != 'https' or signed.expires_in <= 0:
    raise RuntimeError('Expected a cloud S3 presigned download URL.')
try:
    with httpx.stream('GET', signed_url, timeout=httpx.Timeout(120, connect=30), follow_redirects=False) as download:
        if download.status_code != 200:
            raise RuntimeError(f'S3 download failed (HTTP {download.status_code}); request a fresh URL and check bucket access.')
        with partial_path.open('wb') as target:
            for chunk in download.iter_bytes():
                target.write(chunk)
    partial_path.replace(cog_path)
except httpx.RequestError:
    raise RuntimeError('S3 download connection failed; rerun this cell for a fresh URL.') from None
finally:
    partial_path.unlink(missing_ok=True)
    del signed, signed_url
print(f'Downloaded COG: {cog_path.stat().st_size:,} bytes')


Downloaded COG: 4,156 bytes


## Open the COG as an interactive map layer
Rasterio reads the COG and reprojects its first band to a bounded WGS84 preview (at most 1024 pixels per dimension). Positive flooded pixels are blue; dry/nodata pixels are transparent. The full-resolution COG is unchanged. This is a georeferenced image overlay, not a full-resolution tile server. The OpenStreetMap basemap needs internet access; the overlay is embedded locally and does not expose the S3 URL. The kernel/frontend must support ipyleaflet widgets.

In [18]:
import base64
import io
import numpy as np
import rasterio
from rasterio.transform import from_bounds
from rasterio.warp import Resampling, reproject, transform_bounds
from PIL import Image
from ipyleaflet import ImageOverlay, Map, LayersControl, basemaps

with rasterio.open(cog_path) as dataset:
    if dataset.crs is None:
        raise ValueError('The output raster has no CRS and cannot be placed on a map.')
    west, south, east, north = transform_bounds(dataset.crs, 'EPSG:4326', *dataset.bounds, densify_pts=21)
    if not (-180 <= west < east <= 180 and -90 <= south < north <= 90):
        raise ValueError('Invalid or antimeridian-crossing bounds; this preview requires a single geographic extent.')
    scale = min(1, 1024 / max(dataset.width, dataset.height))
    width, height = max(1, round(dataset.width * scale)), max(1, round(dataset.height * scale))
    preview = np.full((height, width), np.nan, dtype=np.float32)
    reproject(
        source=rasterio.band(dataset, 1), destination=preview,
        src_transform=dataset.transform, src_crs=dataset.crs, src_nodata=dataset.nodata,
        dst_transform=from_bounds(west, south, east, north, width, height),
        dst_crs='EPSG:4326', dst_nodata=np.nan, resampling=Resampling.nearest,
        warp_mem_limit=64,
    )

rgba = np.zeros((height, width, 4), dtype=np.uint8)
rgba[np.isfinite(preview) & (preview > 0)] = [30, 110, 255, 180]
png = io.BytesIO()
Image.fromarray(rgba).save(png, format='PNG')
image_uri = 'data:image/png;base64,' + base64.b64encode(png.getvalue()).decode('ascii')
bounds = ((south, west), (north, east))
flood_map = Map(center=((south + north) / 2, (west + east) / 2), zoom=10,
                basemap=basemaps.OpenStreetMap.Mapnik, scroll_wheel_zoom=True)
flood_map.layout.height = '550px'
flood_map.add(ImageOverlay(url=image_uri, bounds=bounds, name='HAND-FIM inundation (COG preview)'))
flood_map.add(LayersControl())
flood_map.fit_bounds(bounds)
display(flood_map)


Map(center=[36.05719377562742, -79.09056623449905], controls=(ZoomControl(options=['position', 'zoom_in_text',…